In [ ]:
%load_ext autoreload
%autoreload 2

# Imports, setup, data loading

In [ ]:
import os
import pandas as pd
from autogluon.tabular import TabularPredictor
from make_clinical_dataset.epr.combine import merge_closest_measurements
from make_clinical_dataset.shared.constants import ROOT_DIR
from ml_common.summary import get_label_distribution
from ml_common.autogluon import train_models, evaluate
pd.set_option('display.max_columns', 100)

In [ ]:
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"

In [ ]:
main = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_data.parquet')
dates = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_dates.parquet')
carg = pd.read_csv('/cluster/projects/grantgroup/CTCAE/filtered_df_with_carg.csv', parse_dates=['date_referred'])
carg = carg[['mrn', 'date_referred', 'carg_toxicity_risk']]

In [ ]:
meta_cols = [
    'mrn', 'assessment_date', 'target_ED_note', 'primary_site_desc', 'study_drug', 'postalcode',
    'target_hemoglobin_min', 'target_platelet_min', 'target_neutrophil_min',
    'target_creatinine_max', 'target_alanine_aminotransferase_max',
    'target_aspartate_aminotransferase_max', 'target_total_bilirubin_max',
]
targ_cols = [col for col in main.columns if col.startswith('target') and col not in meta_cols]
feat_cols = main.columns.drop(meta_cols+targ_cols).tolist()

# GO-TREAT - v2

In [ ]:
root_dir = '/cluster/projects/gliugroup/work_dir/kevin_he'

In [ ]:
# get first treatments only
first_trt_idxs = dates.reset_index().groupby(['mrn','first_treatment_date']).first()['index'].tolist()
main = main.loc[first_trt_idxs]

In [ ]:
# get the develpment and test set
main = merge_closest_measurements(
    main, carg, main_date_col='assessment_date', meas_date_col='date_referred', time_window=(-90,0), 
    merge_individually=False
)
split_date = main['date_referred'].min()
print(f'Temporal split at {split_date}')

very_first_trt = main.groupby('mrn')['assessment_date'].transform('first')
mask = very_first_trt >= split_date
dev, test = main[~mask].copy(), main[mask].copy()
dev = dev[~dev['mrn'].isin(test['mrn'])] # don't include mrns from test set
meta_cols += ['date_referred', 'carg_toxicity_risk']

In [ ]:
dev_meta, dev_target, dev_feats = dev[meta_cols].copy(), dev[targ_cols].copy(), dev[feat_cols].copy()
test_meta, test_target, test_feats = test[meta_cols].copy(), test[targ_cols].copy(), test[feat_cols].copy()
dev_meta['split'] = 'Dev'
test_meta['split'] = 'Test'

X, Y, meta = pd.concat([dev_feats, test_feats]), pd.concat([dev_target, test_target]), pd.concat([dev_meta, test_meta])

In [ ]:
get_label_distribution(Y, meta, with_respect_to='sessions')

In [ ]:
get_label_distribution(Y, meta, with_respect_to='patients')

In [ ]:
models = train_models(
    dev_feats, 
    dev_target.drop(columns=['target_ED_30d', 'target_ED_60d', 'target_ED_90d']), # already computed
    dev_meta, 
    save_path=f'{root_dir}/AutogluonModels/go-treat-v2/'
)

In [ ]:
models = {}
for target in Y.columns:
    model_dir = f'{root_dir}/AutogluonModels/go-treat-v2/{target}-medium-average_precision'
    if os.path.exists(model_dir):
        models[target] = TabularPredictor.load(model_dir, verbosity=0)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

In [ ]:
evaluate(models, test_feats, test_target)

In [ ]:
# AUC of CARG
from ml_common.eval import auc_scores
mask = test_meta['carg_toxicity_risk'].notna()
carg_score = test_meta.loc[mask, 'carg_toxicity_risk'].replace({'Low': '1', 'Moderate': '2', 'High': '3'}).astype(int)
res = {}
for target, label in test_target[mask].items():
    res[target] = auc_scores(label[label != -1], carg_score[label != -1])
pd.DataFrame(res)

In [ ]:
print("Test Data with CARG Score (after anchoring CARG to treatments with a lookback window of 90 days)")
print(f"Pateints = {test_meta.loc[mask, 'mrn'].nunique()}")
print(f"Sessions = {len(test_meta[mask])}")

# GO-TREAT - v1

In [ ]:
print("A2R Data")
print(f"Pateints = {main['mrn'].nunique()}")
print(f"Sessions = {len(main)}")

print("\nCarg Data")
print(f"Pateints = {carg['mrn'].nunique()}")
print(f"Sessions = {len(carg)}")

In [ ]:
print("A2R Data")
print(f"Pateints (Baseline Age >= 65) = {sum(main.groupby('mrn').first()['age'] >= 65)}")

In [ ]:
main.groupby('mrn').first()['assessment_date'].dt.year.value_counts().sort_index().plot(kind='bar')

In [ ]:
carg.groupby('mrn').first()['date_referred'].dt.year.value_counts().sort_index().plot(kind='bar')

In [ ]:
# get the develpment and test set
main = merge_closest_measurements(
    main, carg, main_date_col='assessment_date', meas_date_col='date_referred', time_window=(-90,0), 
    merge_individually=False
)
mask = main['date_referred'].notna()
dev, test = main[~mask].copy(), main[mask].copy()
dev = dev[~dev['mrn'].isin(test['mrn'])] # don't include mrns from test set
meta_cols += ['date_referred', 'carg_toxicity_risk']

In [ ]:
dev_meta, dev_target, dev_feats = dev[meta_cols].copy(), dev[targ_cols].copy(), dev[feat_cols].copy()
test_meta, test_target, test_feats = test[meta_cols].copy(), test[targ_cols].copy(), test[feat_cols].copy()
dev_meta['split'] = 'Dev'
test_meta['split'] = 'Test'

X, Y, meta = pd.concat([dev_feats, test_feats]), pd.concat([dev_target, test_target]), pd.concat([dev_meta, test_meta])

In [ ]:
print("Test Data (after anchoring CARG to treatments with a lookback window of 90 days)")
print(f"Pateints = {test_meta['mrn'].nunique()}")
print(f"Sessions = {len(test_meta)}")

In [ ]:
get_label_distribution(Y, meta, with_respect_to='sessions')

In [ ]:
get_label_distribution(Y, meta, with_respect_to='patients')

In [ ]:
# !python ~/slurm/go-treat.py

In [ ]:
import os
root_dir = '/cluster/projects/gliugroup/work_dir/kevin_he'
models = {}
for target in Y.columns:
    model_dir = f'{root_dir}/AutogluonModels/go-treat-v1/{target}-medium-average_precision'
    if os.path.exists(model_dir):
        models[target] = TabularPredictor.load(model_dir, verbosity=0)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

In [ ]:
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]
pd.concat(res, axis=1)

### Baseline Evaluation

In [ ]:
def evaluate(
    models: dict[str, TabularPredictor],
    X: pd.DataFrame,
    Y: pd.DataFrame,
    return_full: bool = False,
) -> pd.DataFrame:
    """Evaluate performance for all targets and all model types

    Args:
        return_full: If True, return the full information about models (training times, inference times, stack levels, etc)
    """
    results = {}
    for target, model in models.items():
        mask = Y[target] != -1
        if Y.loc[mask, target].nunique() == 1: continue
        
        data = pd.concat([X[mask], Y.loc[mask, target]], axis=1)
        res = model.leaderboard(data, extra_metrics=["roc_auc", "average_precision"])
        results[target] = (
            res if return_full else res[["model", "roc_auc", "average_precision"]]
        )
    results = pd.concat(results, axis=1)
    return results

In [ ]:
baseline_idxs = test_meta.reset_index().groupby('mrn').first()['index'].tolist()

In [ ]:
evaluate(models, test_feats.loc[baseline_idxs], test_target.loc[baseline_idxs])

In [ ]:
# AUC of CARG
from ml_common.eval import auc_scores
carg_score = test_meta.loc[baseline_idxs, 'carg_toxicity_risk'].replace({'Low': '1', 'Moderate': '2', 'High': '3'}).astype(int)
res = {}
for target, label in test_target.loc[baseline_idxs].items():
    mask = label != -1
    res[target] = auc_scores(label[mask], carg_score[mask])
pd.DataFrame(res)